# Lesson 10 Lab — INT8 SmoothQuant and Activation Outliers

**Puzzle:** Can we make activations easier to quantize without changing the floating-point linear layer?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

LLM activations often contain persistent channel outliers that make one tensor-wide INT8 scale waste most of its codes. SmoothQuant does not delete those outliers; it moves part of their range into corresponding weight channels through an exactly equivalent floating-point reparameterization, then quantizes the easier pair.


## 0. Predict before running

1. Prove that reciprocal channel scaling leaves `XWᵀ` unchanged before quantization.
2. Predict why alpha values near either endpoint can hurt combined W8A8 error.
3. Choose the validation metric that should select alpha after calibration.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

SmoothQuant operates on matching input channels of activation `X` and weight `W` for a linear layer `Y=XWᵀ`.

- SmoothQuant applies reciprocal channel scaling to activations and weights, preserving the floating-point product.
- The alpha parameter allocates quantization difficulty between activation and weight channels.
- The best alpha depends on observed activation and weight ranges.


## 2. Derive the mechanism

For positive channel scales `s`, `(X / s)(W · s)ᵀ = XWᵀ`. Choosing `s_j` from activation and weight maxima moves channel difficulty without changing the floating-point function. The exponent `alpha` decides how much range moves toward weights.

For positive channel scales s, define `X' = X / s` and `W' = W · s` along matching input channels. Then `X'W'ᵀ = (X/s)(W·s)ᵀ = XWᵀ`. A common SmoothQuant form constructs s from activation and weight maxima with an exponent alpha, so alpha controls how much range is assigned to each side.

The equality holds before quantization. After W8A8 rounding, shrinking activation outliers reduces activation step size while enlarged weight channels increase weight step size. The objective is the error of the composed quantized linear output, not activation amax in isolation.

### Mechanism at a glance

```mermaid
flowchart LR
  X["Activation X<br/>channel outliers"] --> XS["X' = X / s<br/>smaller activation range"]
  W["Weight W"] --> WS["W' = W · s<br/>absorbs migrated range"]
  XS --> M["Quantized linear path"]
  WS --> M
  M --> Y["Compare with Y = XW^T"]
  A["alpha sweep"] --> S["choose s per channel"]
  S --> XS
  S --> WS
```

### Walk it step by step

1. **Measure channel ranges.** Collect activation and weight maxima on calibration data for matching input channels.
2. **Choose reciprocal scales.** Use alpha to decide how much range moves from each activation channel into its weight channel.
3. **Verify floating equivalence.** Before rounding, confirm that (X/s)(W·s)^T still equals XW^T.
4. **Quantize and validate.** Select alpha by held-out output or task quality, then verify a named W8A8 runtime path.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "10-smoothquant"
device = require_cuda()
torch.manual_seed(2026 + 10)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | W8A8 quantization without activation-to-weight migration (`alpha=0`) |
| Candidate | reciprocal channel scaling for alpha 0.25, 0.5, 0.75, and 1.0 |
| Held constant | same outlier-heavy X and W, per-tensor INT8 reference quantizer, held shapes |
| Measurements | floating-point equivalence max error and quantized output RMSE/cosine by alpha |
| Evidence | `numerical-model` |

**Experiment:** Apply SmoothQuant-style channel scaling to an outlier-heavy linear layer, verify floating-point equivalence, and compare W8A8 reconstruction error over alpha values.


## 5. Read the experiment code

The notebook checks the algebraic invariant before quantizing both sides and comparing output error across alpha values.

The notebook first evaluates the invariant in floating point for every alpha. Only after that check does it quantize both transformed tensors and compare the output with the original FP32 linear layer. This ordering prevents an algebra or broadcasting bug from being mistaken for quantization error.

The sweep uses one calibration-like tensor and reports a numerical model, not a TensorRT-LLM SmoothQuant kernel. A production experiment would freeze scales on calibration data, evaluate held-out tasks, and measure a named W8A8 backend.

Only after these variables match the protocol should the cell be executed.


In [2]:
batch,in_f,out_f=512,512,384; x=torch.randn(batch,in_f,device=device); w=torch.randn(out_f,in_f,device=device)
x[:,::64]*=18; reference=x@w.t(); rows=[]
for alpha in (0.0,0.25,0.5,0.75,1.0):
    ax=x.abs().amax(0).clamp_min(1e-6); aw=w.abs().amax(0).clamp_min(1e-6)
    s=(ax.pow(alpha)/aw.pow(1-alpha)).clamp_min(1e-6); xs=x/s; ws=w*s
    equivalence=(reference-xs@ws.t()).abs().max().item()
    sx=xs.abs().max()/127; sw=ws.abs().max()/127
    qx=torch.round(xs/sx).clamp(-128,127)*sx; qw=torch.round(ws/sw).clamp(-128,127)*sw
    rows.append({"alpha":alpha,"float_equivalence_max_abs":round(equivalence,6),"output_error":error_metrics(reference,qx@qw.t())})
result=base_result(10,"numerical-model"); result.update({"shape":[batch,in_f,out_f],"alpha_sweep":rows,
    "conclusion":"Reciprocal scaling preserved the floating-point layer while changing combined W8A8 error."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Alpha 0 RMSE | 3.298184 |
| Alpha 0.25 RMSE | 1.663379 |
| Alpha 0.5 RMSE | 1.151840 |
| Alpha 0.75 RMSE | 1.634807 |
| Alpha 1 RMSE | 3.224155 |
| Worst floating equivalence error | 0.000061 |


## 7. Interpret rather than merely print

Floating-point equivalence stayed within roughly `6.1e-5` for every alpha. Quantized RMSE followed a U-shape: 3.298184 at alpha 0, 1.663379 at 0.25, a minimum of 1.151840 at 0.5, then 1.634807 at 0.75 and 3.224155 at 1.0. Cosine similarity peaked at 0.999785 for alpha 0.5.

The middle value balanced activation and weight difficulty for this synthetic distribution. The endpoints moved too much error to one side. This supports the migration mechanism while leaving the best alpha model- and layer-dependent.

**Inspection rule:** First verify algebraic equivalence; then compare quantized output error. A lower activation range alone is incomplete evidence.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA numerical experiment isolates an algorithmic mechanism. It is not the paper's complete implementation and does not establish a production kernel speedup.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "alpha_sweep": [
    {
      "alpha": 0.0,
      "float_equivalence_max_abs": 6.1e-05,
      "output_error": {
        "cosine": 0.99824685,
        "mae": 2.62906098,
        "max_abs": 15.94864273,
        "rmse": 3.29818392
      }
    },
    {
      "alpha": 0.25,
      "float_equivalence_max_abs": 4.6e-05,
      "output_error": {
        "cosine": 0.99955261,
        "mae": 1.32819116,
        "max_abs": 7.69526672,
        "rmse": 1.6633786
      }
    },
    {
      "alpha": 0.5,
      "float_equivalence_max_abs": 6.1e-05,
      "output_error": {
        "cosine": 0.99978542,
        "mae": 0.91851175,
        "max_abs": 5.98374176,
        "rmse": 1.15184021
      }
    },
    {
      "alpha": 0.75,
      "float_equivalence_max_abs": 6.1e-05,
      "output_error": {
        "cosine": 0.99956822,
        "mae": 1.30419183,
        "max_abs": 7.36639786,
        "rmse": 1.63480675
      }
    },
    {
      "alpha": 1.0,
      "float_equivalence_max_abs": 4.6e-05,
      "outp

## 9. Make the bounded decision

> Outlier migration is useful only when the combined activation-plus-weight quantized path improves under a frozen calibration protocol.

**Acceptance/rollback:** Verify floating-point equivalence first, freeze calibration statistics, sweep alpha on calibration data, and accept using held-out output/quality plus native W8A8 evidence.

**Failure analysis:** Choosing alpha from the same held-out set used for final quality reporting leaks the test. Reducing activation range without quantizing weights can give a false victory. Another failure is folding scales into weights but forgetting the corresponding activation transform or its runtime/fusion cost.


## 10. Extend the evidence

Freeze channel statistics on one tensor set and select alpha on a separate validation set, then report task quality on a third. Compare per-layer versus global alpha and inspect which layers retain outliers. Finally run a native W8A8 backend and verify that the scale transforms are folded or fused as intended.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
